# The classical secretary problem

**Goal.** Select the single best candidate from $n$ candidates who arrive in uniformly random order. Absolute rank $1$ denotes the best candidate, but only relative ranks are visible during interviews. Rejections are final.

A *record* is better than every candidate interviewed earlier. The threshold strategy rejects the first $r$ candidates, then selects the first later record. If none appears, it loses.

This notebook uses only standard Python tools for simulation; matplotlib is optional for plotting.


## Exact mathematical solution

For $1\le r<n$ the success probability is

$$
P_n(r)=\frac{r}{n}\sum_{j=r}^{n-1}\frac1j,
\qquad P_n(0)=\frac1n.
$$

The smallest maximizing integer $r$ is the finite-$n$ optimum. Both $r/n$ and $P_n(r)$ approach $1/e$ as $n\to\infty$ when $r$ is chosen optimally. For small $n$, do not simply round $n/e$.


In [ ]:
from fractions import Fraction
import random
import math

def exact_probability(n, r):
    if not (isinstance(n, int) and n >= 2 and isinstance(r, int) and 0 <= r < n):
        raise ValueError("Require n >= 2 and 0 <= r < n.")
    if r == 0:
        return Fraction(1, n)
    return Fraction(r, n) * sum((Fraction(1, j) for j in range(r, n)), Fraction(0))

def optimal_threshold(n):
    probabilities = [exact_probability(n, r) for r in range(n)]
    maximum = max(probabilities)
    return probabilities.index(maximum), maximum

n, r = 30, 11
best_r, best_p = optimal_threshold(n)
print(f"Your threshold: r={r}, P={float(exact_probability(n, r)):.5f}")
print(f"Smallest optimal threshold: r={best_r}, P={float(best_p):.5f}")
assert optimal_threshold(10) == (3, Fraction(3349, 8400))


## Monte Carlo experiment

We generate a random permutation of the absolute ranks $1,\ldots,n$. The stopping rule observes the permutation sequentially and selects a new record after the initial $r$ rejections.

The simulated winner is known only after the decision, when we compare the selected absolute rank with $1$. Use a fixed random seed for reproducibility.


In [ ]:
def selected_absolute_rank(order, r):
    if r == 0:
        return order[0]
    best_seen = min(order[:r])
    for rank in order[r:]:
        if rank < best_seen:
            return rank
    return None

def simulate(n, r, trials=10000, seed=2026):
    if trials <= 0:
        raise ValueError("trials must be positive")
    exact_probability(n, r)  # validate parameters
    rng = random.Random(seed)
    successes = 0
    for _ in range(trials):
        order = list(range(1, n + 1))
        rng.shuffle(order)
        successes += (selected_absolute_rank(order, r) == 1)
    return successes / trials

trials = 10000
estimate = simulate(n, r, trials)
print(f"Simulated success: {estimate:.4%} ({trials} repetitions)")
print(f"Exact success:     {float(exact_probability(n, r)):.4%}")
print(f"Absolute error:    {abs(estimate - float(exact_probability(n,r))):.4%}")


## Explore the probability curve

The horizontal axis is the integer rejection threshold. The graph highlights the smallest maximizing threshold and the chosen threshold. Plotting is optional if matplotlib is not available.


In [ ]:
thresholds = list(range(n))
probabilities = [float(exact_probability(n, k)) for k in thresholds]
try:
    import matplotlib.pyplot as plt
    plt.figure(figsize=(9, 4))
    plt.plot(thresholds, probabilities, label="Exact success probability")
    plt.scatter([r], [probabilities[r]], label=f"Chosen r={r}", zorder=3)
    plt.scatter([best_r], [float(best_p)], label=f"Optimal r={best_r}", zorder=3)
    plt.axvline(n / math.e, linestyle="--", label="n/e approximation")
    plt.xlabel("Number initially rejected (r)")
    plt.ylabel("Probability of selecting the best")
    plt.legend()
    plt.show()
except ImportError:
    print("Install matplotlib to view the optional figure.")


## Exercises

1. Set $n=10$. Verify exactly that the smallest optimal threshold is $r=3$.
2. Try $n=50$ and compare $r=1$, $r=\lfloor n/e\rfloor$, and $r=n-1$.
3. Repeat with $1{,}000$ and $20{,}000$ simulations. Which estimates are closer to the exact value?
4. Explain why nonrecord candidates cannot be the overall best.

**Interpretation:** simulation illustrates the theorem; the harmonic sum establishes the exact probability.
